In [ ]:
# ===== PORTFOLIO-124 (idea 12 + mu-descent) — CONFIG (edit ONLY this cell) =
# Race the validated top CoV rankers + reseed_orbit + cov_mu_lex against the
# same-budget greedy control on the 124 unsolved ACA classes. One resumable
# jsonl per budget; resume key = (strategy, pres_id, budget); relaunching a
# dead session is always safe.
#
# MULTI-SESSION CHUNKING: run DIFFERENT strategy lists in different Colab
# sessions (each gets its own GROUP tag -> its own files, zero write races):
#   session 1: STRATEGIES=["cov_mu_lex"],       GROUP="mu"
#   session 2: STRATEGIES=["cov_abel_len_lex"], GROUP="abel"
#   session 3: STRATEGIES=["cov_nsubs_escape"], GROUP="nsubs"
#   session 4: STRATEGIES=["cov_deep_z","cov_defining_iso"], GROUP="deepdef"
#   session 5: STRATEGIES=["reseed_orbit"],     GROUP="reseed"
# (baseline control is force-included in every file automatically)
# HIGH-CAP ARM (T9): a separate session with CAP=48 re-runs the space the
# 24-cap structurally excludes -> new files, do NOT mix caps in one file.

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "research/stable-ac-escape"   # must match the actual git branch
REPO_DIR = "ACSolverX"
UPDATE_REPO = True

MOUNT_DRIVE = True
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx_results/stable_ac_portfolio"

BENCH      = "aca_124"
STRATEGIES = ["cov_mu_lex"]        # see chunking table above
GROUP      = "mu"                  # short tag; part of the file identity
BUDGETS    = [50000]               # one file per budget; 50k first, then escalate
CAP        = 24                    # T9 high-cap arm: 48 (separate session/files)
JOBS       = None                  # None -> cpu_count() - 2
HIGH_SPEEDUP = True                # ~3x, result-neutral, resume-compatible
ROW_LIMIT  = None                  # None = all 124
USE_WANDB  = False                 # jsonl is source of truth; if True, log in via
                                   # a Colab Secret first (never hardcode the key)


In [ ]:
# ==================== SETUP (clone / pull / install) ======================
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy pyyaml")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
else:
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
# a RESTARTED kernel may hold stale modules from before a git reset — purge
for m in [k for k in sys.modules if k == "experiments" or k.startswith("experiments.")]:
    del sys.modules[m]
print("repo root:", REPO_ROOT)


In [ ]:
# ==================== RUN =================================================
os.environ["ACSOLVERX_ALLOW_BIG"] = "1"      # Colab is where big budgets run
from experiments.stable_ac.idea_bench.run_portfolio import run_portfolio

written = run_portfolio(
    bench=BENCH, strategies=STRATEGIES, group=GROUP, budgets=BUDGETS,
    cap=CAP, jobs=JOBS, row_limit=ROW_LIMIT,
    mirror_dir=(DRIVE_DIR if (IN_COLAB and MOUNT_DRIVE) else None),
    high_speedup=HIGH_SPEEDUP, use_wandb=USE_WANDB)
print("files:", written)
# Any SOLVED line above is a LEAD: verify per RUN_ME_MORNING.md before belief.
